# Wrangle raw counts ready for DE analysis

<br>


Select kernel: Single Cell R (4.3)

------

In [ ]:
##### ----- Load libraries
requiredPackages <- c("edgeR","GSEABase", "data.table", "limma", "cowplot", "ggplot2", "dplyr","Matrix", "singscore") #"Manu",ggscireadxl

for (pkg in requiredPackages){
  suppressWarnings(suppressMessages(library(pkg, character.only = T)))
}

fig <- function(width, heigth){
 options(repr.plot.width = width, repr.plot.height = heigth)
 }


##### ----- Get directories
raw_data_path = "/lustre/scratch125/cellgen/behjati/jp40/pipelines/6_RNA/data/counts/"
data_save_dir = "/lustre/scratch125/cellgen/behjati/project_folders/wilms_EMSeq/output/rna/"
sample_sheet_dir = "/lustre/scratch125/cellgen/behjati/project_folders/wilms_EMSeq/metadata/"
                   
sample_sheet_df = read.table(paste0(sample_sheet_dir,"driver_and_manifest_table_2026_07_03_pm.csv"), sep=",", skip=1, header=TRUE)

<br>

### Load

In [ ]:
## -- get file names
files <- list.files(
  raw_data_path,
  pattern = "\\.converted_counts\\.tsv\\.gz$",
  full.names = TRUE
)

In [ ]:
## -- load each sample
read_one_count_file <- function(file) {
  sample_id <- sub("\\.converted_counts\\.tsv\\.gz$", "", basename(file))
  lines <- readLines(gzfile(file))
  header_line <- grep("^#ensid", lines)
  lines[header_line] <- sub("^#", "", lines[header_line])

  dt <- read.delim(
    gzfile(file),
    sep = "\t",
    header = FALSE,
    comment.char = "#",
    stringsAsFactors = FALSE,
    check.names = FALSE
  )
  colnames(dt) =  c("ensid","gene","biotype","chr","longest_isoform","count","unfiltered_count","fpkm","fpkm_uq","tpm")
    
  type_to_include = c("protein_coding", "lincRNA", "miRNA", "misc_RNA", "snRNA", "snoRNA")
  dt <- dt[dt$biotype %in% type_to_include, c("ensid", "unfiltered_count")]

  # Sum duplicate gene symbols if present
  dt <- aggregate(
    unfiltered_count ~ ensid,
    data = dt,
    FUN = sum
  )

  names(dt)[2] <- sample_id

  return(dt)
}

count_list <- lapply(files, read_one_count_file)

In [ ]:
## -- combine all samples in one shot instead of looping merge()
dt_list <- lapply(count_list, as.data.table)
counts_dt <- Reduce(function(a, b) merge(a, b, by = "ensid", all = TRUE), dt_list)

counts_df <- as.data.frame(counts_dt)
counts_df[is.na(counts_df)] <- 0

In [ ]:
write.csv(
  counts_df,
  file = file.path(data_save_dir, "combined_raw_counts.csv")
)

<br>

<br>

### And gene mappings

In [ ]:
## -- build ensid -> gene name mapping, stopping once all genes are covered
build_gene_map_fast <- function(files, target_genes) {
  found <- data.table(ensid = character(), gene = character())
  remaining <- target_genes

  for (file in files) {
    if (length(remaining) == 0) break  # all genes already mapped, stop early

    # count how many leading comment lines (##... and #ensid...) to skip
    peek <- readLines(gzfile(file), n = 20)
    skip_n <- sum(grepl("^#", peek))

    dt <- fread(
      cmd = paste("zcat", shQuote(file)),
      sep = "\t",
      header = FALSE,
      skip = skip_n,
      select = 1:2,
      col.names = c("ensid", "gene")
    )

    dt <- dt[ensid %chin% remaining]
    if (nrow(dt) == 0) next

    found <- rbind(found, unique(dt))
    remaining <- setdiff(remaining, found$ensid)
  }

  unique(found)
}

gene_map <- build_gene_map_fast(files, all_genes)

## -- sanity check
setdiff(all_genes, gene_map$ensid)  # should be character(0)

In [ ]:
all_genes <- sort(unique(unlist(lapply(count_list, function(x) x$ensid))))

missing <- setdiff(all_genes, gene_map$ensid)
length(missing)  # should be 0

<br>

#### Save

In [ ]:
counts_df$gene_name <- as.vector(setNames(gene_map$gene, gene_map$ensid)[counts_df$ensid])

In [ ]:
write.csv(
  counts_df,
  file = file.path(data_save_dir, "combined_raw_counts.csv")
)

<br>

<br>

## 